# Solutions · 07 · Linear viscoelasticity

Worked solutions to the exercises of [notebook 07](../notebooks/07_linear_viscoelasticity.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engrheo import datasets
from engrheo import viscoelastic as ve
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A cross-linked rubber must not creep indefinitely. Which model - Maxwell or standard linear solid - can describe it? Compare their creep compliances after 10 relaxation times.

In [2]:
mx, sls = ve.maxwell(1000.0, 1000.0), ve.standard_linear_solid(300.0, 700.0, 1.0)
for name, m in (("Maxwell", mx), ("standard linear solid", sls)):
    print(f"{name:<22}: J(10 s) = {m['creep'](10.0)*1e3:.2f} 1/kPa, J(100 s) = {m['creep'](100.0)*1e3:.2f} 1/kPa")

Maxwell               : J(10 s) = 11.00 1/kPa, J(100 s) = 101.00 1/kPa
standard linear solid : J(10 s) = 3.22 1/kPa, J(100 s) = 3.33 1/kPa


The Maxwell compliance keeps growing linearly - it flows - whereas the standard linear solid levels off at
$1/G_e$. A cross-linked rubber has a finite equilibrium modulus, so only the solid model (or a spectrum with
$G_e > 0$) can describe it.

## Exercise 2

What fraction of the dough's strain at 120 s has been recovered 180 s after unloading? Compare the measurement, the Burgers prediction and the limit for very long times.

In [3]:
dough = pd.read_csv(datasets.path("dough_creep_recovery.csv"), comment="#")
tt, strain = dough.time_s.to_numpy(), dough.strain.to_numpy()
def history(t_, G1, eta1, G2, eta2):
    J = lambda s: np.where(s >= 0, ve.burgers_creep(np.maximum(s, 0), G1, eta1, G2, eta2), 0.0)
    return 50.0 * (J(t_) - np.where(t_ > 120, J(t_ - 120), 0.0))
p, _ = optimize.curve_fit(history, tt, strain, p0=[4e3, 5e5, 1e3, 1e4], bounds=(0, np.inf))
s120, s300 = strain[tt == 120][0], strain[-1]
permanent = 50 * 120 / p[1]
print(f"recovered after 180 s: measured {(s120 - s300)/s120:.1%}, Burgers {(s120 - history(np.array([300.0]), *p)[0])/s120:.1%}, "
      f"long-time limit {(s120 - permanent)/s120:.1%}")

recovered after 180 s: measured 85.1%, Burgers 85.2%, long-time limit 85.3%


After three minutes the dough has recovered almost all of the strain it ever will: the long-time limit is
barely higher, because the delayed elasticity (retardation time 25 s) has largely come back. The rest - about
15 % of the strain at 120 s - is permanent viscous flow.

## Exercise 3

**Implement it yourself:** fit a single Maxwell element to relaxation data by a straight line of $\ln G$ against $t$ (generate the data from `ve.maxwell(2000, 500)` with 2 % noise). How accurate are $G$ and $\tau$?

In [4]:
rng = np.random.default_rng(7)
mx = ve.maxwell(2000.0, 500.0)
t = np.linspace(0.01, 1.0, 40)
G = mx["relaxation"](t) * np.exp(rng.normal(0, 0.02, t.size))
slope, icpt = np.polyfit(t, np.log(G), 1)
print(f"G = {np.exp(icpt):.0f} Pa (true 2000), tau = {-1/slope:.4f} s (true {mx['tau']:.4f})")

G = 1989 Pa (true 2000), tau = 0.2497 s (true 0.2500)


For a single exponential the logarithm is a straight line, so ordinary linear regression gives both
parameters to about a percent. With several modes the log plot curves and this trick no longer works - the
spectrum methods of notebook 09 take over.